# Fruit & Vegetable Classification — Colab Training

Trains the same Custom CNN, ResNet50, and VGG16 architectures as `fruit-vegetable-classification-dual.ipynb` on the Fruit and Vegetable Image Recognition dataset. Checkpoints are saved in `models/` for use with the local Streamlit app (`streamlit_app.py`) and its shared inference module (`fruit_vegetable_models.py`).

Run the cells in order. The KaggleHub dataset is public; no Kaggle credentials are needed. Set `DATA_DIR_OVERRIDE` if you already have the dataset in Drive. Training all three models can take several hours; use a Colab GPU and edit `MODEL_NAMES` to train only selected models.

In [ ]:
%pip -q install kagglehub scikit-learn


In [ ]:
import json
import time
import zipfile
from pathlib import Path

import kagglehub
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Set this to a folder containing train/, validation/, and test/, or keep None to download.
DATA_DIR_OVERRIDE = None
if DATA_DIR_OVERRIDE is None:
    downloaded_path = Path(kagglehub.dataset_download('kritikseth/fruit-and-vegetable-image-recognition'))
    DATA_DIR = downloaded_path
else:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)

def find_dataset_root(base):
    split_names = ('train', 'validation', 'test')
    if all((base / name).is_dir() for name in split_names):
        return base
    for train_folder in base.rglob('train'):
        candidate = train_folder.parent
        if all((candidate / name).is_dir() for name in split_names):
            return candidate
    raise FileNotFoundError(f'Could not find train/validation/test under {base}')

DATA_DIR = find_dataset_root(DATA_DIR)
TRAIN_DIR, VAL_DIR, TEST_DIR = (DATA_DIR / name for name in ('train', 'validation', 'test'))
OUTPUT_DIR = Path('models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset: {DATA_DIR.resolve()}')

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 5
NUM_WORKERS = 2
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_ds = datasets.ImageFolder(VAL_DIR, transform=eval_transform)
test_ds = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
CLASSES = train_ds.classes
for split_name, dataset in (('validation', val_ds), ('test', test_ds)):
    if dataset.classes != CLASSES:
        raise ValueError(f'{split_name} class order/names do not match training classes')
NUM_CLASSES = len(CLASSES)

loader_options = {'num_workers': NUM_WORKERS, 'pin_memory': DEVICE.type == 'cuda'}
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, **loader_options)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_options)
print(f'{NUM_CLASSES} classes | train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}')
print(CLASSES)

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_channels, out_channels):
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(
            conv_block(3, 64), conv_block(64, 128),
            conv_block(128, 256), conv_block(256, 512),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.4), nn.Linear(256, num_classes),
        )

    def forward(self, image):
        return self.classifier(self.features(image))

def build_model(model_name):
    if model_name == 'custom_cnn':
        return CustomCNN(NUM_CLASSES)
    if model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.fc.in_features, NUM_CLASSES))
        return model
    if model_name == 'vgg16':
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[6] = nn.Sequential(nn.Dropout(0.4), nn.Linear(4096, NUM_CLASSES))
        return model
    raise ValueError(f'Unknown model: {model_name}')

# Edit this list to train only some architectures.
MODEL_NAMES = ['custom_cnn', 'resnet50', 'vgg16']

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct = 0.0, 0
    all_labels, all_predictions = [], []
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            predictions = logits.argmax(dim=1)
            total_loss += loss.item() * images.size(0)
            correct += (predictions == labels).sum().item()
            all_labels.extend(labels.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())
    count = len(loader.dataset)
    return total_loss / count, correct / count, all_labels, all_predictions

def train_model(model_name):
    model = build_model(model_name).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    checkpoint_path = OUTPUT_DIR / f'{model_name}_best.pth'
    best_accuracy, epochs_without_improvement = -1.0, 0

    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        train_loss, train_accuracy, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_accuracy, _, _ = run_epoch(model, val_loader, criterion)
        scheduler.step(val_accuracy)
        print(
            f'{model_name} | {epoch:02d}/{NUM_EPOCHS} | '
            f'train loss {train_loss:.4f} acc {train_accuracy:.4f} | '
            f'val loss {val_loss:.4f} acc {val_accuracy:.4f} | {time.time() - started:.1f}s'
        )
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            epochs_without_improvement = 0
            torch.save({
                'model_state': model.state_dict(),
                'classes': CLASSES,
                'model_name': model_name,
                'image_size': IMG_SIZE,
                'mean': list(MEAN),
                'std': list(STD),
            }, checkpoint_path)
            print(f'Saved best checkpoint: {checkpoint_path} (val acc {best_accuracy:.4f})')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f'Early stopping {model_name} at epoch {epoch}')
                break
    return checkpoint_path

CHECKPOINTS = {name: train_model(name) for name in MODEL_NAMES}

In [ ]:
for model_name, checkpoint_path in CHECKPOINTS.items():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    model = build_model(model_name).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    _, test_accuracy, labels, predictions = run_epoch(model, test_loader, nn.CrossEntropyLoss())
    print(f'\n{model_name} test accuracy: {test_accuracy:.4f}')
    print(classification_report(
        labels, predictions, labels=list(range(NUM_CLASSES)),
        target_names=CLASSES, zero_division=0, digits=4,
    ))

archive_path = Path('fruit_vegetable_models.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for checkpoint_path in CHECKPOINTS.values():
        archive.write(checkpoint_path, arcname=f'models/{checkpoint_path.name}')
print(f'Model archive ready: {archive_path.resolve()}')

# In Colab, download the trained checkpoints for local/Streamlit use.
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print('Not running in Colab; find checkpoints in the models/ folder.')